# 🫁 Explanation-Supervised Attention — Inference Server

Starts a **Flask inference API** that your local `dashboard.html` Live Demo connects to.

### Every session — run cells in this order:
1. **Cell 1** — install packages & mount Drive
2. **Cell 2** — copy project from Drive + list checkpoints
3. **Cell 3** — extract `src/` from Drive zip
4. **Cell 4** — start Flask server + ngrok
5. Copy the **ngrok URL** → paste into dashboard Live Demo → Server URL field

> Keep this notebook running while using the dashboard.

In [ ]:
# ── Cell 1: Install packages & mount Drive ──────────────────────────────────
!pip install flask flask-cors pyngrok pillow requests --quiet

from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print('✓ Google Drive mounted')

In [ ]:
# ── Cell 2: Set paths & copy project from Drive ─────────────────────────────
import os, sys, shutil

# ── Edit this if your Drive folder name is different ────────────────────────
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/DL_Project_Outputs'
CKPT_SUBDIR       = 'checkpoints'   # checkpoints live directly here
# ────────────────────────────────────────────────────────────────────────────

COLAB_ROOT = '/content/dl_project'
if not os.path.exists(COLAB_ROOT):
    shutil.copytree(DRIVE_PROJECT_DIR, COLAB_ROOT)
    print(f'✓ Project copied to {COLAB_ROOT}')
else:
    print(f'✓ Already at {COLAB_ROOT}')

sys.path.insert(0, COLAB_ROOT)
CKPT_DIR = os.path.join(COLAB_ROOT, CKPT_SUBDIR)

print('\nCheckpoint directory:', CKPT_DIR)
if os.path.exists(CKPT_DIR):
    for f in sorted(os.listdir(CKPT_DIR)):
        mb = os.path.getsize(os.path.join(CKPT_DIR, f)) / 1e6
        print(f'  {f}  ({mb:.1f} MB)')
else:
    print('  ⚠ Not found — check DRIVE_PROJECT_DIR above')

In [ ]:
# ── Cell 3: Extract src/ from Drive zip (needed every session) ──────────────
import zipfile, shutil, os

ZIP_PATH = '/content/drive/MyDrive/dl-project-code.zip'

if not os.path.exists(ZIP_PATH):
    print(f'⚠ Zip not found at {ZIP_PATH}')
    print('  Check your Drive or re-upload dl-project-code.zip')
else:
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall('/content/dl_project_temp')

    for root, dirs, files in os.walk('/content/dl_project_temp'):
        if 'src' in dirs:
            src_from = os.path.join(root, 'src')
            src_to   = '/content/dl_project/src'
            if os.path.exists(src_to):
                shutil.rmtree(src_to)
            shutil.copytree(src_from, src_to)
            print(f'✓ src/ ready at {src_to}')
            break

    print('Contents:', os.listdir('/content/dl_project/src'))

In [ ]:
# ── Cell 4: Start Flask inference server + ngrok ─────────────────────────────
import io, base64, subprocess, threading
import numpy as np
import torch
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok

from src.models.model import build_model
from src.gradcam import GradCAM, get_gradcam_layer

# Fix DenseNet Grad-CAM layer (norm5 has in-place op conflicts)
import src.gradcam as _gcam
_gcam.GRADCAM_LAYERS['densenet121'] = 'backbone.features.denseblock4'

# ── Constants ────────────────────────────────────────────────────────────────
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)
CLASS_NAMES   = [
    'Atelectasis','Cardiomegaly','Effusion','Infiltration','Mass',
    'Nodule','Pneumonia','Pneumothorax','Consolidation','Edema',
    'Emphysema','Fibrosis','Pleural Thickening','Hernia'
]
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

# ── Model cache ──────────────────────────────────────────────────────────────
_model_cache = {}

def get_model(backbone: str, variant: bool):
    key = f'{backbone}_{"attn" if variant else "base"}'
    if key not in _model_cache:
        cfg   = {'backbone': backbone, 'pretrained': False,
                 'num_classes': 14, 'use_channel_attn': True}
        model = build_model(cfg, cooc_matrix=None, variant=variant)
        tag   = f'{backbone}_{"attention" if variant else "baseline"}_best.pt'
        ckpt_path = os.path.join(CKPT_DIR, tag)
        if os.path.exists(ckpt_path):
            ckpt  = torch.load(ckpt_path, map_location='cpu', weights_only=False)
            state = ckpt.get('model', ckpt)
            model.load_state_dict(state, strict=False)
            print(f'  ✓ Loaded: {tag}')
        else:
            print(f'  ⚠ No checkpoint: {tag} — random weights')
        model.to(DEVICE).eval()
        _model_cache[key] = model
    return _model_cache[key]

# ── Helpers ───────────────────────────────────────────────────────────────────
def preprocess(pil_img):
    tf = T.Compose([
        T.Resize((224, 224)), T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])
    return tf(pil_img.convert('RGB')).unsqueeze(0).to(DEVICE)

def map_to_list(arr):
    mn, mx = arr.min(), arr.max()
    return ((arr - mn) / (mx - mn + 1e-8)).tolist()

def img_to_b64(pil_img):
    buf = io.BytesIO()
    pil_img.save(buf, format='PNG')
    return base64.b64encode(buf.getvalue()).decode()

# ── Flask app ─────────────────────────────────────────────────────────────────
app = Flask(__name__)
CORS(app)

@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'ok', 'device': DEVICE})

@app.route('/predict', methods=['POST'])
def predict():
    try:
        if 'image' not in request.files:
            return jsonify({'error': 'No image provided'}), 400

        backbone = request.form.get('backbone', 'resnet50')
        mode     = request.form.get('mode', 'variant')
        variant  = (mode == 'variant')
        thresh   = float(request.form.get('threshold', 0.5))

        pil_img = Image.open(request.files['image']).convert('RGB')
        img_224 = pil_img.resize((224, 224))
        x       = preprocess(pil_img)
        model   = get_model(backbone, variant)

        with torch.no_grad():
            logits, attn_map = model(x)

        probs   = torch.sigmoid(logits).squeeze().cpu().numpy()
        attn_np = attn_map.squeeze().cpu().numpy()
        attn_up = F.interpolate(
            attn_map.cpu(), size=(224,224), mode='bilinear', align_corners=False
        ).squeeze().numpy()

        # Grad-CAM from baseline model
        base_model = get_model(backbone, variant=False)
        layer      = get_gradcam_layer(backbone)
        gcam       = GradCAM(base_model, layer)
        top_cls    = int(probs.argmax())
        gcam_224   = gcam(x, class_idx=top_cls, output_size=(224,224)).squeeze()
        gcam.remove_hooks()

        detections = sorted(
            [{'class': CLASS_NAMES[i], 'prob': float(probs[i]),
              'detected': bool(probs[i] >= thresh)} for i in range(14)],
            key=lambda d: -d['prob']
        )

        return jsonify({
            'status': 'ok', 'backbone': backbone, 'mode': mode,
            'top_class': CLASS_NAMES[top_cls], 'top_prob': float(probs[top_cls]),
            'detections': detections,
            'attn_map':   map_to_list(attn_np),
            'attn_224':   map_to_list(attn_up),
            'gradcam_224':map_to_list(gcam_224),
            'image_b64':  img_to_b64(img_224),
        })
    except Exception as e:
        import traceback
        return jsonify({'error': str(e), 'trace': traceback.format_exc()}), 500

# ── Free port 5000 if still in use ───────────────────────────────────────────
subprocess.run(['fuser', '-k', '5000/tcp'], capture_output=True)

# ── ngrok ─────────────────────────────────────────────────────────────────────
# Paste your free ngrok auth token from ngrok.com (required)
NGROK_AUTH_TOKEN = ''   # ← paste your token here

if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
ngrok.kill()
public_url = ngrok.connect(5000).public_url

print('\n' + '='*60)
print('  ✅ Inference server is LIVE')
print(f'  ngrok URL  →  {public_url}')
print('='*60)
print('  Paste URL into dashboard Live Demo → Server URL field')
print('  Keep this notebook running while using the demo.')
print('='*60 + '\n')

print('Pre-warming ResNet-50 attention model...')
get_model('resnet50', variant=True)
print('✓ Ready\n')

threading.Thread(
    target=lambda: app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False),
    daemon=True
).start()

In [ ]:
# ── Cell 5 (optional): Health check ─────────────────────────────────────────
import requests as req
print(req.get(f'{public_url}/health').json())